# **Build a Mini-BERT Model From Scratch**

In [2]:
#Import Libraries

import torch
import torch.nn as nn 
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB
from transformers import BertTokenizer
import requests
from zipfile import ZipFile
from torch.utils.data import Dataset
import pandas as pd
import json
from torchtext.functional import pad_sequence
from torch.utils.data import DataLoader
from tensorflow import Tensor
import math
import ast


C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
train_iter, val_iter = IMDB()


In [4]:
#define tokenizor
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [5]:
def download (url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(response.content)
    else:
        print(f"Failed to download. Status code: {response.status_code}")

In [6]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/bZaoQD52DcMpE7-kxwAG8A.zip'
filename = 'BERT-dataset'

In [7]:
download(url, filename)

In [8]:
def unzip_file (filename,directory):
    with ZipFile(filename, 'r') as unzip_f:
        unzip_f.extractall(directory)

In [5]:
directory = 'D:\Projects\mini-bert-nsp-mlm-pretraining'

In [10]:
unzip_file(filename,directory)

In [6]:
data = pd.read_csv('bert_dataset/bert_train_data.csv')
data.iloc[2]

Original Text    I rented I AM CURIOUS-YELLOW from my video sto...
BERT Input       [1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...
BERT Label       [0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...
Segment Label    1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...
Is Next                                                          1
Name: 2, dtype: object

In [7]:
row =  data.iloc[2]
ans = torch.tensor(ast.literal_eval(row["BERT Input"]))
print(ans.type)

<built-in method type of Tensor object at 0x00000207170EF830>


In [8]:
data.head()

,Original Text,BERT Input,BERT Label,Segment Label,Is Next
0,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 16, 94, 12615, 5, 1026, 22, 5, 201, 18, 11...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",0
1,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 16, 1571, 16, 249, 35471, 3, 67, 3, 1138, ...","[0, 0, 0, 0, 0, 0, 46, 0, 401, 0, 0, 0, 0, 5, ...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1
2,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...","[0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1
3,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 9266, 54, 14, 3, 783, 11, 2483, 17, 3, 7, ...","[0, 0, 0, 0, 134, 0, 0, 0, 0, 685, 7, 0, 0, 9,...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",0
4,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 66, 14519, 4444, 7, 4637, 76, 1479, 11, 60...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 60, 0, 0, 7552, 0,...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1


### **Build Dataset Class**

In [9]:
class BertDatatsetCSV(Dataset):
    def __init__(self, filename):

        self.dataset = pd.read_csv(filename)
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def __len__(self):
        return len(self.dataset)

    def safe_json(self, x):
        if isinstance(x, str):
            return json.loads(x)
        return x

    def __getitem__(self, idx):
        self.dataset = self.dataset.reset_index(drop=True)
        row = self.dataset.iloc[idx]

        bert_input = torch.tensor(
            ast.literal_eval(str(row["BERT Input"]).strip()), dtype=torch.long
        )

        bert_label = torch.tensor(
            ast.literal_eval(str(row["BERT Label"]).strip()), dtype=torch.long
        )

        segment_label = torch.tensor(
            ast.literal_eval(str(row["Segment Label"]).strip()), dtype=torch.long
        )

        is_next = torch.tensor(row["Is Next"], dtype=torch.long)

        return bert_input, bert_label, segment_label, is_next

In [10]:
row =  data.iloc[31360]
bert_input = torch.tensor(json.loads(row['BERT Input']), dtype=torch.long)
bert_input

tensor([   1,   49,   12,   19,   47,   84,  365,  135,    6,    2,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,   21, 1040,
           7,    5,    3,    3,   94,  135,    3, 1142,    3,   22,  294, 8429,
           7,    3,  143,    3,   34,   32,    3,    2])

In [11]:
data = data.reset_index(drop=True)
row = data.iloc[3]

bert_input = torch.tensor(
        ast.literal_eval(str(row["BERT Input"]).strip()), dtype=torch.long
    )
bert_input

tensor([    1,  9266,    54,    14,     3,   783,    11,  2483,    17,     3,
            7,  1578,   121,     3,   345,    10,   117,  1163,     3,    16,
           75,    78,     3,    77,     3,    22,   540,     6,     2,     0,
           15,   206,  2185,  7274,     3,  1922,     3,    10, 21481,    53,
            3,  4659,    31,  2384,     7,    64,    55,   405,    23,    50,
          477,  1695,     7,  8138,     7,     8,  1002,     3,     6,     2])

### **Build Collate Function**

In [12]:
PAD_IDX = 0

def collate_func(batch):
    bert_input_batch = []
    bert_label_batch = []
    segment_label_batch = []
    is_next_batch = []

    for bert_input, bert_label, segment_label, is_next in batch:
        bert_input_batch.append(bert_input)
        bert_label_batch.append(bert_label)
        segment_label_batch.append(segment_label)
        is_next_batch.append(is_next)

    bert_input_batch = pad_sequence(
        bert_input_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    bert_label_batch = pad_sequence(
        bert_label_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    segment_label_batch = pad_sequence(
        segment_label_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    is_next_batch = torch.tensor(is_next_batch, dtype=torch.long)

    return bert_input_batch, bert_label_batch, segment_label_batch, is_next_batch

### **Initialize the Dataset and Dataloaders**

In [13]:
train_dataset = BertDatatsetCSV("./bert_dataset/bert_train_data.csv")
test_dataset = BertDatatsetCSV("./bert_dataset/bert_test_data.csv")

In [14]:
train_dataset.__len__()

170540

In [15]:
test_dataset.__len__()

167449

In [16]:
batch_size = 4

In [17]:
train_dataloader = DataLoader(train_dataset,batch_size=batch_size, collate_fn=collate_func,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size, collate_fn=collate_func)

### **Model Implementation**

Model Creation (BERT Embeddings)

In BERT, three types of embeddings are combined to represent each input token:

1. Token Embedding

This is the basic representation of each word/token.

* Maps every token to a fixed-size dense vector
* Learns semantic meaning of words
* Captures relationships between tokens based on context

👉 Think of it as: “What does this word mean?”

2. Positional Embedding

Transformers process tokens in parallel, so they don’t naturally understand order.

* Adds position information to each token
* Encodes where a token appears in the sequence
* Helps the model understand word order and structure

👉 Think of it as: “Where is this word in the sentence?”

3. Segment Embedding

Used when BERT processes multiple sentences (e.g., sentence pairs).

* Assigns different embeddings to different segments (Sentence A, Sentence B)
* Helps distinguish between parts of the input
* Useful for tasks like question answering and next sentence prediction

👉 Think of it as: “Which sentence does this word belong to?”

Final Input Representation

The final embedding for each token is the sum of all three:

* Final Embedding = Token Embedding + Positional Embedding + Segment Embedding

This combined representation allows BERT to understand meaning, order, and context simultaneously.

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

#### **Token Embeddings**

In [19]:
class TokenEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, num_dims):
        super().__init__()
        self.num_dims = num_dims
        self.embeddings = nn.Embedding(vocab_size, num_dims)
    
    def forward(self, input_tokens:Tensor):
        embedding_out = self.embeddings(input_tokens.long())
        return embedding_out * math.sqrt(self.num_dims)

#### **Position Embedding**

In [20]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        positions = torch.arange(0, max_seq_len).unsqueeze(1)

        pe = torch.zeros((max_seq_len, d_model))

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)

        pe = pe.unsqueeze(1)
        self.register_buffer("positional_encodings", pe)

    def forward(self, word_embeddings):
        # word_embeddings -> [seq_len, batch_size, num_dims]
        seq_len = word_embeddings.size(0)
        positional_embeddings = (
            word_embeddings + self.positional_encodings[0:seq_len, :, :]
        )
        return self.dropout(positional_embeddings)

#### **BERT Embeddings**

In [49]:
class BERTEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, d_models,max_seq_len, dropout):
        super().__init__()
        self.token_embeddings = TokenEmbeddings(vocab_size, d_models)
        self.positional_embeddings = PositionalEmbeddings(max_seq_len,d_models,dropout)
        self.segment_embeddings = nn.Embedding(3,d_models)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self,word_tokens, segment_ids):
        embed_out = self.token_embeddings(word_tokens)
        pos_embed = self.positional_embeddings(embed_out)
        segment_embed = self.segment_embeddings(segment_ids)
        
        bert_embeddings = pos_embed + segment_embed
        return self.dropout(bert_embeddings)

#### **Initiate Sample Dataset for Testing**

In [50]:
count = 0

for batch in train_dataloader:
    bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
    
    count += 1
    if count == 5:
        break

### **BERT Model Architecture Overview**

* The complete BERT model consists of the following key components:

1. Initialization

The BERT class extends torch.nn.Module. During initialization, it defines core hyperparameters such as:

* Vocabulary size
* Hidden/model dimension
* Number of encoder layers
* Number of attention heads
* Dropout rate

These parameters configure the overall architecture of the model.

2. Embedding Layer

The embedding layer is implemented using the BERTEmbedding class. It combines:

* Token embeddings
* Segment embeddings

This layer converts input tokens into dense vector representations suitable for the transformer.

3. Transformer Encoder

A stack of Transformer Encoder layers processes the embeddings. Each layer applies:

* Multi-head self-attention
* Feedforward neural networks

The depth (number of layers), number of attention heads, model dimension, and dropout are controlled by the initialized parameters.

4. Next Sentence Prediction (NSP) Head

A linear classification layer is used for the NSP task. It:

* Takes the encoder output (typically the [CLS] token representation)
* Predicts whether two sentences are consecutive
* Outputs probabilities for two classes (IsNext / NotNext)

5. Masked Language Modeling (MLM) Head

Another linear layer handles the MLM task. It:

* Uses encoder outputs for all token positions
* Predicts the original tokens for masked positions
* Produces probability distributions over the vocabulary

6. Forward Pass

The forward method defines how data flows through the model:

* Inputs: bert_inputs, segment_labels
* Process: Embedding → Transformer Encoder → Task-specific heads

Outputs:
* NSP predictions
* MLM predictions

In [77]:
class BERTModel (nn.Module):
    
    def __init__(self, vocab_size, d_model, max_seq_len, n_layers, n_heads, dropout = 0.1):
        
        super().__init__()
        #define bert embedding
        self.bert_embedding = BERTEmbeddings(vocab_size,d_model,max_seq_len,dropout)
        # define encoder layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, 
                                                        dropout=dropout, batch_first=True)
        # define encoder
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,num_layers=n_layers)
        
        #define linear layer for mask language modeling
        self.fc_mlm = nn.Linear(d_model, vocab_size)
        
        #define linear layer for next word prediction
        self.fc_nsp = nn.Linear(d_model, 2)
        
    def forward(self,bert_inputs, segment_labels):
        bert_out = self.bert_embedding(bert_inputs, segment_labels)
        
        #we need to create padding mask to prevent attention to padding
        pad_mask = (bert_inputs == PAD_IDX).transpose(0,1)
        
        encoder_out = self.encoder(bert_out, src_key_padding_mask = pad_mask)
        
        # for mlm
        mlm_out = self.fc_mlm(encoder_out)
        
        #for nsp
        nsp_out = self.fc_nsp(encoder_out[:,0,:])
        
        return mlm_out, nsp_out

#### **Create a Instance of the Model**

In [78]:
vocab_size = 147161
d_model = 12
n_layers = 2
nheads = 4
dropout = 0.1

In [79]:
bert_model = BERTModel(vocab_size,d_model,500,n_layers,nheads,dropout)

In [80]:
padding_mask = (bert_inputs == PAD_IDX).transpose(0,1)
padding_mask.shape

torch.Size([4, 46])

In [81]:
bert_inputs.shape

torch.Size([46, 4])

In [82]:
bert_inputs[:,0]

tensor([    1,     3,    22,   157, 65455,     3,   177,    41,     5,  2173,
            6,     2,     0,     0,     0,     0,    77,     5,  5644,   206,
        24843,     8,     5,    88,     3,     5,  1312,     3,     5, 11924,
           39,     2,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0])

In [83]:
token_embddings = TokenEmbeddings(vocab_size,d_model)

t_embeddings = token_embddings(bert_inputs)
t_embeddings.shape


torch.Size([46, 4, 12])

In [84]:
positional_encoding = PositionalEmbeddings(max_seq_len=500,d_model=d_model,dropout=dropout)
pe = positional_encoding(t_embeddings)
pe.shape    

torch.Size([46, 4, 12])

In [85]:
for i in range(3):
    print(f"postional embddings  for the {i}th token of the first sample: {pe[i,0,:]}")

postional embddings  for the 0th token of the first sample: tensor([-2.5599, -1.6212,  4.6280, -4.6542, -1.7341, -0.1322, -0.5349, -0.0000,
         1.7665,  0.0755,  7.5265,  3.0468], grad_fn=<SliceBackward0>)
postional embddings  for the 1th token of the first sample: tensor([ 9.7691,  0.8588, -4.6507, -2.6560,  4.0163, -0.2010, -5.0133, -4.2918,
        -2.8711, -6.1348, -6.2673, -1.9160], grad_fn=<SliceBackward0>)
postional embddings  for the 2th token of the first sample: tensor([ 4.5426, -1.4220, -0.0000,  7.1612, -2.0378,  3.6171,  2.9609, -1.3255,
        -0.2618, -3.8895,  4.3508, -0.4440], grad_fn=<SliceBackward0>)


In [86]:
segment_embedding = nn.Embedding(3,d_model)
se = segment_embedding(segment_labels)
se.shape

torch.Size([46, 4, 12])

In [87]:
for i in range(3):
    print(f"segment embddings  for the {i}th token of the first sample: {se[i,0,:]}")

segment embddings  for the 0th token of the first sample: tensor([-0.5829,  0.6696,  0.9233, -0.6148,  0.0975, -1.0540, -0.0133,  0.2651,
        -0.5223,  0.7565,  0.2923,  0.8126], grad_fn=<SliceBackward0>)
segment embddings  for the 1th token of the first sample: tensor([-0.5829,  0.6696,  0.9233, -0.6148,  0.0975, -1.0540, -0.0133,  0.2651,
        -0.5223,  0.7565,  0.2923,  0.8126], grad_fn=<SliceBackward0>)
segment embddings  for the 2th token of the first sample: tensor([-0.5829,  0.6696,  0.9233, -0.6148,  0.0975, -1.0540, -0.0133,  0.2651,
        -0.5223,  0.7565,  0.2923,  0.8126], grad_fn=<SliceBackward0>)


In [88]:
bert_embeddings = pe + se
bert_embeddings.shape

torch.Size([46, 4, 12])

In [89]:
for i in range(3):
    print(f"bert embddings  for the {i}th token of the first sample: {bert_embeddings[i,0,:]}")

bert embddings  for the 0th token of the first sample: tensor([-3.1429, -0.9516,  5.5513, -5.2690, -1.6366, -1.1862, -0.5482,  0.2651,
         1.2443,  0.8320,  7.8188,  3.8594], grad_fn=<SliceBackward0>)
bert embddings  for the 1th token of the first sample: tensor([ 9.1862,  1.5284, -3.7274, -3.2708,  4.1138, -1.2549, -5.0266, -4.0267,
        -3.3933, -5.3783, -5.9751, -1.1034], grad_fn=<SliceBackward0>)
bert embddings  for the 2th token of the first sample: tensor([ 3.9596, -0.7524,  0.9233,  6.5464, -1.9402,  2.5631,  2.9476, -1.0604,
        -0.7841, -3.1330,  4.6430,  0.3686], grad_fn=<SliceBackward0>)


#### **Idea About the Encdoing Layers (Optional)**

In [90]:
encoding_layer = nn.TransformerEncoderLayer(d_model,nheads,dropout=dropout, batch_first=False)
transformer_encoder = nn.TransformerEncoder(encoder_layer=encoding_layer, num_layers=n_layers)

#pass the bert embeddings to transformer encoder
te = transformer_encoder(bert_embeddings, src_key_padding_mask = padding_mask)
te.shape

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


torch.Size([46, 4, 12])

In [91]:
for i in range(3):
    print(f"transformer encoder  for the {i}th token of the first sample: {te[i,0,:]}")

transformer encoder  for the 0th token of the first sample: tensor([-0.7134, -0.4166,  0.4720, -1.6246,  0.1602, -0.4272, -1.3643, -0.3493,
        -0.0025,  1.2934,  1.7810,  1.1913], grad_fn=<SliceBackward0>)
transformer encoder  for the 1th token of the first sample: tensor([ 2.3511,  0.7626,  0.0441, -1.0191,  1.4233, -0.1686, -0.7498, -0.7831,
        -0.8073, -0.8777,  0.2145, -0.3900], grad_fn=<SliceBackward0>)
transformer encoder  for the 2th token of the first sample: tensor([ 1.0124, -0.8732,  1.0326,  0.2034, -0.6867,  0.9781, -0.0139,  0.5448,
        -2.3083, -0.2995,  1.1740, -0.7636], grad_fn=<SliceBackward0>)


##### **Next Sentence Prediction**

In [92]:
nsp_layer = nn.Linear(d_model,2)
nsp = nsp_layer(te[0,:])
nsp.shape

torch.Size([4, 2])

In [93]:

print(nsp)

tensor([[-0.3140, -1.2959],
        [-0.0113, -0.3289],
        [-0.2193, -0.9101],
        [ 0.0033, -0.7465]], grad_fn=<AddmmBackward0>)


##### **Mask Language Modeling**

In [94]:
mlm_layer = nn.Linear(d_model, vocab_size)
mlm = mlm_layer(te)

mlm.shape

torch.Size([46, 4, 147161])

In [95]:
print(mlm[0:3,:,:])

tensor([[[ 0.0808,  0.6234, -0.1950,  ..., -1.0606,  0.1587, -0.0058],
         [ 0.2064,  0.7302,  0.9838,  ..., -0.6886,  0.4643, -0.4891],
         [ 0.5180,  0.7944,  0.4938,  ..., -1.0946,  0.2663, -0.1717],
         [ 0.2376,  0.3766,  0.5256,  ..., -1.1124,  0.2785, -0.1341]],

        [[ 0.2926, -0.2193,  1.3380,  ..., -0.3839,  0.2568, -0.2758],
         [-0.4817, -0.1068,  0.8842,  ..., -0.9244,  0.1618,  0.7055],
         [ 0.0132, -0.0563,  0.9975,  ...,  0.7120,  0.9926, -1.0614],
         [-0.1091,  0.1174,  0.4605,  ..., -0.0280,  0.5119, -0.3049]],

        [[-0.5955, -0.9760,  0.5183,  ..., -0.9346,  0.1167,  0.2101],
         [ 0.0749,  0.2915, -0.4250,  ..., -0.8796, -0.2784, -0.0126],
         [-0.4510,  0.5524, -0.8174,  ...,  0.5667,  0.7589, -0.5706],
         [-0.1300, -0.4335, -0.7172,  ..., -1.0977,  0.4043, -0.4114]]],
       grad_fn=<SliceBackward0>)


In [96]:
bert_model = bert_model.to(device)

## **Model Evaluation, Training and Prediction**

In [97]:
# The loss function must ignore PAD tokens and only calculates loss for the masked tokens
mlm_loss = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
nsp_loss = nn.CrossEntropyLoss()

### **Model Evaluation** 

In [ ]:
def model_eval(model:BERTModel, dataloader:DataLoader, nsp_loss = nsp_loss, mlm_loss = mlm_loss, device=device):
    
    model.eval()
    
    nsp_loss_val = 0
    mlm_loss_val = 0
    total_batches = 0
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
            
            #predict the values from the model
            mlm_out_predict, nsp_out_predict = model(bert_inputs, segment_labels)
            
            # calculate loss for each prediction
            mlm_loss_batch = mlm_loss(mlm_out_predict.view(-1,mlm_out_predict.size(-1)),bert_labels.view(-1))
            
            nsp_loss_batch = nsp_loss(nsp_out_predict, is_nexts.view(-1))
            
            batch_loss = mlm_loss_batch + nsp_loss_batch
            
            if torch.isnan(batch_loss):
                continue
            else:
                total_loss += batch_loss
                nsp_loss_val += nsp_loss_batch
                mlm_loss_val += mlm_loss_batch
                
                total_batches += 1
    
    avg_loss = total_loss / (total_batches + 1)
    avg_nsp_loss = nsp_loss_val / (total_batches + 1)
    avg_mlm_loss = mlm_loss_val / (total_batches + 1)
    
    print(f"Average Loss: {avg_loss:.4f}, Average Next Sentence Loss: {avg_nsp_loss:.4f}, Average Mask Loss: {avg_mlm_loss:.4f}")
    return avg_loss